# NSM Training Agent على Google Colab

هذا الدفتر يشغّل **وكيل تدريب Neural Service Mesh** داخل Colab مع GPU (إن وُجد).

## الخطوات
1. **Runtime → Change runtime type → GPU** (مثلاً T4)
2. نفّذ الخلايا بالترتيب
3. (اختياري) اربط Google Drive لحفظ النماذج

> لا يتحكم هذا الدفتر في Colab من المتصفح الخارجي — الوكيل يعمل **داخل** الجلسة.


In [ ]:
# 0) تفعيل سياسة GPU لـ NSM
import os
os.environ["NSM_ALLOW_GPU"] = "1"
os.environ["NSM_OFFLINE_MODE"] = "0"
print("NSM_ALLOW_GPU =", os.environ["NSM_ALLOW_GPU"])


In [ ]:
# 1) استنساخ المستودع (إن لم يكن موجوداً)
import os
from pathlib import Path

REPO = "Neural-Service-Mesh"
URL = "https://github.com/aliahmed369000000-ai/Neural-Service-Mesh.git"

if not Path(REPO).is_dir():
    !git clone --depth 1 {URL}
else:
    print("المستودع موجود — تحديث اختياري:")
    !git -C {REPO} pull --ff-only || true

%cd {REPO}
print("cwd:", os.getcwd())


In [ ]:
# 2) تثبيت تبعيات التدريب الخفيفة
# Colab عادةً يوفّر torch المتوافق مع CUDA للجلسة
import sys, subprocess

def pipi(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pipi("numpy")
try:
    import torch
    print("torch", torch.__version__, "| cuda?", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("تثبيت torch…", e)
    pipi("torch")
    import torch
    print("torch", torch.__version__, "cuda?", torch.cuda.is_available())


In [ ]:
# 3) (اختياري) ربط Google Drive لحفظ النماذج والبيانات
MOUNT_DRIVE = False  # غيّر إلى True عند الحاجة

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/NSM_Training")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive جاهز:", DRIVE_ROOT)
else:
    print("تم تخطي Drive — النماذج تُحفظ تحت artifacts/model_training/")


In [ ]:
# 4) فحص GPU عبر وكيل NSM
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))

from ai.gpu_runtime import device_report_md, detect_device, suggest_batch_size

print(device_report_md())
info = detect_device(force_gpu=True)
print("الجهاز المفروض للتدريب:", info)
print("batch مقترح لـ 1000 عيّنة:", suggest_batch_size(1000, 64, free_vram_gb=info.free_vram_gb))


In [ ]:
# 5) تدريب تجريبي (CSV العيّنة) عبر الوكيل
from ai.model_training_agent import train_from_csv, handle_training_command

print(handle_training_command("حالة gpu"))
print("---")
print(train_from_csv("data/samples/classification_demo.csv", epochs=15))


In [ ]:
# 6) هدف مصنع مرتبط بـ CKG (بيانات من معرفة المشروع)
from ai.training_factory import run_factory

print(run_factory("هدف: حسّن تصنيف كيانات CKG بدقة أعلى من 70%"))


In [ ]:
# 7) (اختياري) نسخ أفضل النماذج إلى Google Drive
import shutil
from pathlib import Path

if MOUNT_DRIVE:
    src = Path("artifacts/model_training")
    dst = DRIVE_ROOT / "artifacts_model_training"
    dst.mkdir(parents=True, exist_ok=True)
    for p in src.glob("*.pt"):
        shutil.copy2(p, dst / p.name)
        print("نُسخ:", p.name)
    print("المجلد:", dst)
else:
    print("فعّل MOUNT_DRIVE = True في الخلية 3 ثم أعد التشغيل لنسخ النماذج.")


## ملاحظات
- إذا ظهرت `cuda_not_available`: تأكد من نوع بيئة التشغيل = GPU ثم أعد تشغيل الجلسة.
- `prefer_cpu_for_toy` في المشروع يتجاوزه `NSM_ALLOW_GPU=1`.
- عند OOM يقلّص الوكيل `batch_size` تلقائياً ويعيد المحاولة.
- الطريقة الثانية (التحكم من المتصفح الخارجي عبر Playwright) غير مضمّنة هنا عمداً: هشة ومخالفة غالباً لشروط الاستخدام.
